In [1]:
import joblib
import numpy as np
import pandas as pd

BUNDLE_PATH = "stack_lgbm_iso_bundle.joblib"

In [2]:
def clean_and_fix_temporal(df):
    df = df.copy()

    # --- sale_date (approx from Year+Month if you have them) ---
    # If you also have day, use it. Otherwise pivot to middle-of-month.
    df["SALEDATE_Year"] = pd.to_numeric(df["SALEDATE_Year"], errors="coerce")
    df["SALEDATE_MonthofYearNumber"] = pd.to_numeric(df["SALEDATE_MonthofYearNumber"], errors="coerce").fillna(6)
    df["sale_date"] = pd.to_datetime(df["VRSALEDATE"].astype(str),format="%Y%m%d", errors="coerce")    
    df['IsEV'] = pd.to_numeric(df["IsEV"])


    # --- Vehicle condition grade: extract number safely ---
    # Works for "Grade 3.5", "3.5", 0.0, etc.
    if "Vehicle_condition_overall" in df.columns:
        vc = df["Vehicle_condition_overall"].astype(str).str.extract(r'(\d+(\.\d+)?)')[0]
        df["Vehicle_condition_overall"] = pd.to_numeric(vc, errors="coerce")

    # --- Numeric coercions you rely on downstream ---
    num_cols = [
        'VRMILEAGE','Vehicle_year','Vehicle_cylinders','Vehicle_doors',
        'Vehicle_engine','Vehicle_condition_overall','EngineHP','BasePrice',
        'SALEDATE_WeekofYearNumber','SALEDATE_MonthofYearNumber','SALEDATE_Quarter',
        'SALEDATE_Year','vehicle_age','mileage_per_year','log_mileage','log_age',
        'drivable_flag','GVWR_class'
    ]
    for c in num_cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors="coerce")

    # --- clamp & age at sale ---
    df["Vehicle_year"] = df["Vehicle_year"].clip(1900, 2100)
    df["vehicle_age"] = (df["SALEDATE_Year"] - df["Vehicle_year"]).clip(lower=0)

    # --- mileage hygiene ---
    # cap ridiculous mileage and fix negatives
    df["VRMILEAGE"] = df["VRMILEAGE"].clip(lower=0, upper=df["VRMILEAGE"].quantile(0.9995))

    # --- safe ratios/logs ---
    age_safe = df["vehicle_age"].replace(0, 0.5)
    df["mileage_per_year"] = df["VRMILEAGE"] / age_safe
    df["log_mileage"] = np.log1p(df["VRMILEAGE"])
    df["log_age"] = np.log1p(df["vehicle_age"])

    # --- drivable ---
    df["drivable_flag"] = (df["Vehicle_condition_drivable"] == "Y").astype(np.int8)

    # --- GVWR numeric class extraction (keeps signal, shrinks OHE) ---
    # "Class 1: 6,000 lb..." -> 1
    if "GVWR" in df.columns:
        df["GVWR_class"] = (
            df["GVWR"].astype(str).str.extract(r'Class\s*(\d+)')[0].astype(float)
        )

    # downcast numerics
    for c in df.select_dtypes(include=[np.number]).columns:
        df[c] = pd.to_numeric(df[c], downcast="float")

    return df

In [3]:
def apply_te_maps(df: pd.DataFrame, te_maps: dict, high_card: list) -> pd.DataFrame:
    df = df.copy()
    for col in high_card:
        g = float(te_maps[col]["global_mean"])
        m = te_maps[col]["mapping"]
        out_col = f"te__{col}"

        if col not in df.columns:
            df[out_col] = g
        else:
            s = df[col].astype("object").fillna("__MISSING__")
            df[out_col] = s.map(m).fillna(g)

        df[out_col] = pd.to_numeric(df[out_col], errors="coerce").fillna(g).astype("float32")
    return df

In [4]:
def make_features(df_clean: pd.DataFrame, bundle: dict) -> pd.DataFrame:
    df = apply_te_maps(df_clean, bundle["te_maps"], bundle["high_card"])
    needed = bundle["numeric_feats"] + bundle["categorical_feats"]
    X = df[needed].copy()

    for c in bundle["numeric_feats"]:
        X[c] = pd.to_numeric(X[c], errors="coerce").astype("float32")
    for c in bundle["categorical_feats"]:
        X[c] = X[c].astype("object")
    return X

In [5]:
import numpy as np
from sklearn.base import BaseEstimator, RegressorMixin, clone

class LogTargetRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, base_estimator):
        self.base_estimator = base_estimator

    def fit(self, X, y):
        self.est_ = clone(self.base_estimator)
        y_log = np.log1p(np.asarray(y, dtype=float))
        self.est_.fit(X, y_log)
        return self

    def predict(self, X):
        y_log_pred = self.est_.predict(X)
        return np.expm1(y_log_pred)


In [6]:
# Load (trusted bundle only) :contentReference[oaicite:6]{index=6}
bundle = joblib.load(BUNDLE_PATH)

In [7]:
df_new = pd.read_excel("Test Customer Data.xlsx")
df_new = clean_and_fix_temporal(df_new)  # you must import or include this function

X_new = make_features(df_new, bundle)

In [8]:
p_stack = bundle["stack_model"].predict(X_new)
p_lgbm  = bundle["lgbm_model"].predict(X_new)

P = np.column_stack([p_stack, p_lgbm])
p_blend = P @ bundle["blend_w"]
pred    = bundle["iso"].transform(p_blend)

df_new["PredictedSalePrice"] = pred
df_new.to_excel("Scored_New_Customer_Data_Stack.xlsx", index=False)

C:\Users\ChenChen\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.13_qbz5n2kfra8p0\LocalCache\local-packages\Python313\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but LGBMRegressor was fitted with feature names
  warnings.warn(
